# Ploemeur single-date example

This notebook is the beginner-friendly entry point for the single-date `ploemeur` example.

Default behavior:
- focus on the observed data,
- run the workflow,
- read the Metropolis-Hastings result first,
- keep the method comparison for an optional expert section.

If you want the detailed comparison with `forward_uncertainty_quantification`, set `expert_mode = True`.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
for parent in [ROOT, *ROOT.parents]:
    if (parent / 'pyproject.toml').exists() and (parent / 'pyage').exists():
        ROOT = parent
        break

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from IPython.display import Image, Markdown, display
import matplotlib.pyplot as plt
import pandas as pd
import pyage.concentrations.concentrations as co
import pyage.global_parameters as gp
import pyage.calibration.utils.systematic_sampling as calibration_exploration
from pyage.lpm.lpm_build import lpm_build
from scripts.common.example_summary_plots import (
    plot_objective_summary,
    plot_parameter_summary,
    plot_single_date_model_space,
)
from scripts.common.launcher_params import load_params
from scripts.launcher import run_workflow

def read_tsv(path):
    frame = pd.read_csv(path, sep='	')
    return frame.loc[:, ~frame.columns.str.startswith('Unnamed')]

def read_stats(path):
    return pd.read_csv(path, sep='	', index_col=0)

print('ROOT:', ROOT)

## Notebook mode

Keep `expert_mode = False` for a first pass.
The workflow still computes the other method outputs, but the notebook will not surface them unless you ask for them.

In [ ]:
expert_mode = False

params_path = ROOT / 'examples' / 'ploemeur' / 'exemple_ploemeur.yaml'
params = load_params(ROOT, params_path)

print('Config:', params_path)
print('Dataset:', params.dataset_name)
print('Reference year:', params.dataset_year)
print('LPM:', params.lpm_model_name)
print('Metropolis-Hastings steps:', params.mh_nstep)
print('Expert mode:', expert_mode)

## Step 1 - Look at the observed concentrations

This example contains one sampling date and three tracers.
Because the dataset is small, the key question is easy to state: where does the observed point sit relative to the reachable concentration space and the calibrated models?

In [ ]:
dataset_path = params.dataset_data_dir / params.dataset_name
cdata = co.Concentrations(file_load=True, file_name=str(dataset_path))

observed = cdata.cv[['element', 'concentration', 'unit', 'date']].copy()
observed['element'] = observed['element'].str.upper()
display(observed)

## Step 2 - Run the workflow

The launcher writes the full result tree.
In this notebook we will then rebuild three simplified MH-only views for a first reading.

In [ ]:
results_dir = run_workflow(params_path, force_inline=True)
results_dir = Path(results_dir)
print('Results directory:', results_dir)

## Step 3 - Beginner view: read the Metropolis-Hastings result first

These notebook figures are intentionally simpler than the full workflow outputs:
- only the Metropolis-Hastings posterior is shown,
- the method comparison is hidden,
- the goal is to understand the result before comparing methods.

In [ ]:
notebook_dir = results_dir / 'notebook_views'
notebook_dir.mkdir(parents=True, exist_ok=True)

mh_dist = read_tsv(results_dir / 'Metropolis_Hastings' / 'lpm_dist_calibrated.txt')
reachable_frame = read_tsv(results_dir / 'reachable_concentrations' / 'c_reach.txt')
param_names = lpm_build(params.lpm_model_name, directory_lpm=str(params.directory_lpm)).get_param_names()
posterior_results = {'Metropolis_Hastings': mh_dist}

objective_sampler = calibration_exploration.SystematicSampling(
    params.lpm_model_name,
    cdata.names(),
    date=cdata.cv['date'],
    cdata=cdata,
    nmodels=params.objective_function_nmodels,
    display_options=gp.display_options(),
    objfunc=True,
    reachconc=False,
)
objective_sampler.compute_concentrations()
objective_sampler.objective_function_build()
objective_frame = objective_sampler.objective_function_frame()

fig = plot_single_date_model_space(
    concentration_sampled=cdata,
    reachable_frame=reachable_frame,
    posterior_results=posterior_results,
    filename=notebook_dir / '01_mh_data_model_space.png',
    title='Ploemeur: observation, reachable space and MH-calibrated models',
)
plt.show()
plt.close(fig)
display(Markdown("Observation, reachable space and calibrated models: the black point is the measured sample, the pale cloud is the reachable space, and the blue points show where the MH-calibrated models concentrate.\n\nIf the posterior cloud stays close to the observation, the calibration is reproducing the data in a coherent part of the feasible domain."))

fig = plot_parameter_summary(
    posterior_results,
    param_names=param_names,
    filename=notebook_dir / '02_mh_parameter_summary.png',
    title='Ploemeur: Metropolis-Hastings parameter distributions',
)
plt.show()
plt.close(fig)
display(Markdown("Metropolis-Hastings parameter distributions: each panel shows where the posterior density accumulates for one parameter.\n\nA narrow distribution means the data constrain that parameter well; a broad distribution means the calibration keeps more uncertainty."))

fig = plot_objective_summary(
    objective_frame=objective_frame,
    posterior_results=posterior_results,
    param_names=param_names,
    filename=notebook_dir / '03_mh_objective_summary.png',
    title='Ploemeur: objective landscape with MH estimates',
)
plt.show()
plt.close(fig)
display(Markdown("Objective landscape with MH estimates: the background color comes from the gridded objective function, and the blue samples show where the calibrated posterior sits in parameter space.\n\nThe useful question here is whether the MH posterior is concentrated near the low-objective region, rather than wandering far from it."))

## Step 4 - Read the MH parameter table

After the figures, the next useful object is the MH statistics table.
Start with the count, mean and standard deviation, then look at the best row.

In [ ]:
mh_stats = read_stats(results_dir / 'Metropolis_Hastings' / 'lpm_stats_calibrated.txt')
display(mh_stats.loc[['count', 'mean', 'std']])

best_row = mh_dist.sort_values('obj_function').iloc[0][param_names + ['obj_function']]
display(Markdown('### Best Metropolis-Hastings sample'))
display(best_row.to_frame('value'))

## Expert mode (optional)

Set `expert_mode = True` only after you are comfortable with the MH result.
This section exposes the comparison with `forward_uncertainty_quantification` and the full workflow comparison figures.

In [ ]:
if expert_mode:
    display(Markdown('### Comparison figures from the full workflow'))
    explanations = {
        '01_data_model_space.png': (
            'Comparison view: the orange and blue samples show how the two calibration methods populate the feasible concentration space.',
            'Use this figure only after the MH-only reading: it answers whether both methods land in the same region of the data space.'
        ),
        '02_parameter_summary.png': (
            'Comparison view of parameter distributions across methods.',
            'This helps identify whether both methods agree on the constrained region of each parameter or whether they tell different stories.'
        ),
        '03_objective_summary.png': (
            'Comparison view of the objective landscape with both calibrated parameter clouds.',
            'The key question is whether both methods identify the same low-objective region, and how concentrated each method remains around it.'
        ),
    }
    for path in [
        results_dir / '01_data_model_space.png',
        results_dir / '02_parameter_summary.png',
        results_dir / '03_objective_summary.png',
    ]:
        if path.exists():
            display(Markdown(f'#### {path.name}'))
            display(Image(filename=str(path)))
            if path.name in explanations:
                text = explanations[path.name]
                display(Markdown(text[0] + '\n\n' + text[1]))

    display(Markdown('### Method statistics'))
    for method in ['forward_uncertainty_quantification', 'Metropolis_Hastings']:
        stats = read_stats(results_dir / method / 'lpm_stats_calibrated.txt')
        display(Markdown(f'#### {method}'))
        display(stats.loc[['count', 'mean', 'std']])
else:
    print('Set expert_mode = True and re-run this cell to display the method comparison.')